# WMDP Classifier-Gated PoRT Mini GPU Test

This notebook validates the canonical `llm-unlearn-eco/scripts/evaluate_wmdp.py` entrypoint in classifier-gated mode from a Kaggle-style clean clone. It requires a local Hugging Face text-classification model directory mounted on Kaggle and exposed through `WMDP_CLASSIFIER_PATH` or `MANUAL_CLASSIFIER_PATH`.

The run uses `multiple_choice_zero_out.yaml`, `sample_size=2`, and intentionally does not pass `--attack_all_prompts`.

In [ ]:
from pathlib import Path
import importlib
import json
import os
import subprocess
import sys

REPO_URL = 'https://github.com/toanthangO20/PoRT_LLM_Unlearning-Experiment.git'
REPO_DIR_NAME = 'PoRT_LLM_Unlearning-Experiment'
IS_KAGGLE = Path('/kaggle/working').exists()

def has_project_layout(path):
    path = Path(path)
    return (path / 'llm-unlearn-eco' / 'eco').exists() and (path / 'dataset').exists()

def clone_or_use_project():
    if IS_KAGGLE:
        target = Path('/kaggle/working') / REPO_DIR_NAME
        if has_project_layout(target):
            print(f'Using existing cloned repository: {target}')
            subprocess.check_call(['git', '-C', str(target), 'pull', '--ff-only'])
            return target.resolve()
        if target.exists():
            raise RuntimeError(f'{target} exists but does not look like this repo.')
        print(f'Cloning {REPO_URL} into {target}')
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
        return target.resolve()

    local_root = Path.cwd().resolve()
    if has_project_layout(local_root):
        return local_root
    target = local_root / REPO_DIR_NAME
    if has_project_layout(target):
        return target.resolve()
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
    return target.resolve()

PROJECT_ROOT = clone_or_use_project()
ECO_ROOT = PROJECT_ROOT / 'llm-unlearn-eco'
os.environ['PORT_PROJECT_ROOT'] = str(PROJECT_ROOT)
if str(ECO_ROOT) not in sys.path:
    sys.path.insert(0, str(ECO_ROOT))

commit_sha = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
print(f'Project root: {PROJECT_ROOT}')
print(f'Commit: {commit_sha}')

In [ ]:
required_packages = {
    'datasets': 'datasets>=2.10.1',
    'pandas': 'pandas',
    'pyarrow': 'pyarrow>=10',
    'safetensors': 'safetensors',
    'tabulate': 'tabulate',
    'tqdm': 'tqdm',
    'transformers': 'transformers>=4.38.0',
    'accelerate': 'accelerate>=0.26.0',
    'yaml': 'pyyaml',
}

missing_packages = []
for module_name, package_spec in required_packages.items():
    try:
        importlib.import_module(module_name)
    except ModuleNotFoundError:
        missing_packages.append(package_spec)

if missing_packages:
    print('Installing missing packages:', missing_packages)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])
else:
    print('Required packages are already available.')

In [ ]:
import platform
import torch

print('Python:', platform.python_version())
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('This classifier-gated smoke test is intended for a Kaggle GPU runtime.')
for idx in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(idx)
    print(f'GPU {idx}: {props.name}, VRAM={props.total_memory / 1024**3:.2f} GiB')

## Resolve Classifier Artifact

Set `MANUAL_CLASSIFIER_PATH` if the Kaggle input path is known. If it is blank, the notebook uses `WMDP_CLASSIFIER_PATH`, then `PORT_WMDP_CLASSIFIER_PATH`, then tries to find one matching Hugging Face model directory under `/kaggle/input`.

In [ ]:
MANUAL_CLASSIFIER_PATH = ''  # Example: '/kaggle/input/wmdp-classifier/model'
CLASSIFIER_MODEL_NAME = os.environ.get('PORT_WMDP_CLASSIFIER_MODEL_NAME', 'wmdp_classifier_llama_guard_3_1b_v2')
CLASSIFIER_THRESHOLD = float(os.environ.get('PORT_WMDP_CLASSIFIER_THRESHOLD', '0.999'))
CLASSIFIER_BATCH_SIZE = int(os.environ.get('PORT_CLASSIFIER_BATCH_SIZE', '1'))

WEIGHT_FILENAMES = {
    'model.safetensors',
    'model.safetensors.index.json',
    'pytorch_model.bin',
    'pytorch_model.bin.index.json',
}
TOKENIZER_FILENAMES = {
    'tokenizer.json',
    'tokenizer_config.json',
    'vocab.json',
    'vocab.txt',
    'spiece.model',
}

def looks_like_hf_model_dir(path):
    path = Path(path)
    if not (path / 'config.json').exists():
        return False
    has_tokenizer = any((path / name).exists() for name in TOKENIZER_FILENAMES)
    has_weights = any((path / name).exists() for name in WEIGHT_FILENAMES)
    has_sharded_weights = bool(list(path.glob('model-*-of-*.safetensors'))) or bool(list(path.glob('pytorch_model-*-of-*.bin')))
    return has_tokenizer and (has_weights or has_sharded_weights)

def find_classifier_candidates(root):
    root = Path(root)
    if not root.exists():
        return []
    candidates = []
    for config_path in root.rglob('config.json'):
        candidate = config_path.parent
        if looks_like_hf_model_dir(candidate):
            score_text = str(candidate).lower()
            score = int('wmdp' in score_text) + int('classifier' in score_text)
            candidates.append((score, candidate))
    candidates.sort(key=lambda item: (-item[0], len(str(item[1]))))
    return [candidate for _, candidate in candidates]

classifier_path_value = (
    MANUAL_CLASSIFIER_PATH.strip()
    or os.environ.get('WMDP_CLASSIFIER_PATH', '').strip()
    or os.environ.get('PORT_WMDP_CLASSIFIER_PATH', '').strip()
)

if not classifier_path_value and IS_KAGGLE:
    candidates = find_classifier_candidates('/kaggle/input')
    if len(candidates) == 1:
        classifier_path_value = str(candidates[0])
        print(f'Auto-detected classifier path: {classifier_path_value}')
    elif candidates:
        print('Multiple classifier-like model directories found:')
        for candidate in candidates[:20]:
            print('  ', candidate)
        raise RuntimeError('Set MANUAL_CLASSIFIER_PATH to the intended WMDP classifier directory.')

if not classifier_path_value:
    raise RuntimeError('Set WMDP_CLASSIFIER_PATH or MANUAL_CLASSIFIER_PATH to a local Hugging Face classifier directory.')

CLASSIFIER_PATH = Path(classifier_path_value).expanduser().resolve()
if not CLASSIFIER_PATH.exists():
    raise FileNotFoundError(f'Classifier path does not exist: {CLASSIFIER_PATH}')
if not CLASSIFIER_PATH.is_dir():
    raise NotADirectoryError(f'Classifier path must be a directory: {CLASSIFIER_PATH}')
if not looks_like_hf_model_dir(CLASSIFIER_PATH):
    print('Classifier directory contents:')
    for child in sorted(CLASSIFIER_PATH.iterdir()):
        print('  ', child.name)
    raise RuntimeError('Classifier path does not look like a local Hugging Face model directory.')

os.environ['WMDP_CLASSIFIER_PATH'] = str(CLASSIFIER_PATH)
print(f'WMDP_CLASSIFIER_PATH={CLASSIFIER_PATH}')
print(f'Classifier model name: {CLASSIFIER_MODEL_NAME}')
print(f'Classifier threshold: {CLASSIFIER_THRESHOLD}')

## Load Classifier Once

In [ ]:
from eco.attack import PromptClassifier
from eco.dataset import WMDPBio
from eco.utils import delete_model

bio = WMDPBio(sample_size=1)
wmdp_prompt = bio.load_dataset_for_eval('test')[0]['prompt']
probe_prompts = [
    wmdp_prompt,
    'What is the capital of France? Answer:',
]

classifier = PromptClassifier(
    model_name=CLASSIFIER_MODEL_NAME,
    model_path=str(CLASSIFIER_PATH),
    batch_size=CLASSIFIER_BATCH_SIZE,
)
try:
    labels = classifier.predict(probe_prompts, threshold=CLASSIFIER_THRESHOLD)
    print('Classifier probe labels:', labels)
    if all(label == 0 for label in labels) or all(label == 1 for label in labels):
        print('WARNING: probe labels are all identical; inspect threshold and label mapping before full runs.')
finally:
    delete_model(classifier)

## Run Canonical Script

In [ ]:
TARGET_CONFIG_NAME = os.environ.get('PORT_TARGET_CONFIG_NAME', 'phi-1_5')
TARGET_HF_NAME = os.environ.get('PORT_TARGET_HF_NAME')
TARGET_MODEL_PATH = os.environ.get('PORT_TARGET_MODEL_PATH')
ATTN_IMPLEMENTATION = os.environ.get('PORT_ATTN_IMPLEMENTATION', 'eager')
TORCH_DTYPE = os.environ.get('PORT_TORCH_DTYPE', 'float16')
BATCH_SIZE = int(os.environ.get('PORT_WMDP_BATCH_SIZE', '1'))
SAMPLE_SIZE = int(os.environ.get('PORT_WMDP_SAMPLE_SIZE', '2'))
TASK_CONFIG = os.environ.get('PORT_WMDP_TASK_CONFIG', 'multiple_choice_zero_out.yaml')
RUN_NAME = os.environ.get(
    'PORT_RUN_NAME',
    f'wmdp_script_mini_zero_out_classifier_gated_{TARGET_CONFIG_NAME}',
)

if IS_KAGGLE:
    print(f'Refreshing repository at {PROJECT_ROOT}')
    subprocess.check_call(['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only'])

SCRIPT_PATH = PROJECT_ROOT / 'llm-unlearn-eco' / 'scripts' / 'evaluate_wmdp.py'
OUTPUT_DIR = Path('/kaggle/working') if IS_KAGGLE else PROJECT_ROOT / 'results'
script_source = SCRIPT_PATH.read_text(encoding='utf-8')
required_script_markers = ['--attack_all_prompts', 'resolve_classifier_path', 'WMDP_CLASSIFIER_PATH']
missing_script_markers = [marker for marker in required_script_markers if marker not in script_source]
if missing_script_markers:
    raise RuntimeError(
        'The cloned repository does not contain the classifier-gated evaluate_wmdp.py updates yet. '
        f'Missing markers: {missing_script_markers}. Pull or push the latest repo commit.'
    )

SCRIPT_ENV = os.environ.copy()
existing_pythonpath = SCRIPT_ENV.get('PYTHONPATH')
SCRIPT_ENV['PYTHONPATH'] = str(ECO_ROOT) if not existing_pythonpath else str(ECO_ROOT) + os.pathsep + existing_pythonpath
SCRIPT_ENV['PORT_PROJECT_ROOT'] = str(PROJECT_ROOT)
SCRIPT_ENV['WMDP_CLASSIFIER_PATH'] = str(CLASSIFIER_PATH)

cmd = [
    sys.executable,
    str(SCRIPT_PATH),
    '--model_name', TARGET_CONFIG_NAME,
    '--task_config', TASK_CONFIG,
    '--sample_size', str(SAMPLE_SIZE),
    '--batch_size', str(BATCH_SIZE),
    '--torch_dtype', TORCH_DTYPE,
    '--attn_implementation', ATTN_IMPLEMENTATION,
    '--classifier_path', str(CLASSIFIER_PATH),
    '--classifier_model_name', CLASSIFIER_MODEL_NAME,
    '--classifier_threshold', str(CLASSIFIER_THRESHOLD),
    '--output_dir', str(OUTPUT_DIR),
    '--run_name', RUN_NAME,
]
if TARGET_HF_NAME:
    cmd.extend(['--target_hf_name', TARGET_HF_NAME])
if TARGET_MODEL_PATH:
    cmd.extend(['--model_path', TARGET_MODEL_PATH])

print('$ ' + ' '.join(cmd))
subprocess.check_call(cmd, cwd=str(PROJECT_ROOT), env=SCRIPT_ENV)

## Verify Artifacts

In [ ]:
import pandas as pd

run_dir = OUTPUT_DIR / RUN_NAME
summary_path = run_dir / 'summary.json'
predictions_path = run_dir / 'predictions.csv'
attack_stats_path = run_dir / 'attack_stats.csv'
summary_by_run_path = run_dir / 'summary_by_run.csv'

for path in [summary_path, predictions_path, attack_stats_path, summary_by_run_path]:
    if not path.exists():
        raise FileNotFoundError(path)

summary = json.loads(summary_path.read_text(encoding='utf-8'))
predictions = pd.read_csv(predictions_path)
attack_stats = pd.read_csv(attack_stats_path)
summary_by_run = pd.read_csv(summary_by_run_path)

expected_rows = SAMPLE_SIZE * 3
if len(predictions) != expected_rows:
    raise AssertionError(f'Expected {expected_rows} prediction rows, got {len(predictions)}')

required_attack_columns = {'run_label', 'dataset', 'classifier_mode', 'num_prompts', 'num_attacked', 'attack_rate'}
missing_columns = required_attack_columns - set(attack_stats.columns)
if missing_columns:
    raise AssertionError(f'Missing attack_stats columns: {sorted(missing_columns)}')
if set(attack_stats['classifier_mode']) != {'classifier_gated'}:
    raise AssertionError(f'Unexpected classifier modes: {sorted(set(attack_stats["classifier_mode"]))}')

print('Prediction rows:', len(predictions))
print('Attack stats:')
print(attack_stats.to_string(index=False))
print('Summary by run:')
print(summary_by_run.to_string(index=False))

rates = attack_stats['attack_rate']
if (rates == 0).all() or (rates == 1).all():
    print('WARNING: attack_rate is all 0 or all 1; debug threshold or classifier labels before full WMDP.')

print('CLASSIFIER-GATED MINI TEST COMPLETED')